In [42]:
import pandas as pd

df = pd.read_csv("CAR_data_V2.csv")

/var/folders/h7/c_5v7d7s0w7d_6fry6l74vyc0000gn/T/ipykernel_50376/3768534756.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("CAR_data_V2.csv")


## Callback instances, picked-up callbacks, pickup rates, callback retries 

In [43]:
# --- Step 1: Ensure proper datetime format ---
df['Activity Start Timestamp'] = pd.to_datetime(df['Activity Start Timestamp'], errors='coerce')

# --- Step 2: Identify sessions that include the Courtesy Callback Telephony EP ---
callback_sessions = (
    df.groupby('Contact Session ID')['EP Name']
      .apply(lambda x: (x == "Courtesy Callback Telephony EP").any())
      .reset_index()
      .rename(columns={'EP Name': 'has_callback_ep'})
)

# --- Step 3: Keep only Contact Session IDs that include a callback EP ---
callbacks = df[df['Contact Session ID'].isin(
    callback_sessions[callback_sessions['has_callback_ep']]['Contact Session ID']
)].copy()

# --- Step 4: Find the first timestamp of the callback EP within each session ---
first_callback_time = (
    callbacks[callbacks['EP Name'] == "Courtesy Callback Telephony EP"]
    .groupby('Contact Session ID')['Activity Start Timestamp']
    .min()
    .reset_index()
    .rename(columns={'Activity Start Timestamp': 'first_callback_time'})
)

# --- Step 5: Merge and keep only rows at or after the callback start ---
callbacks = callbacks.merge(first_callback_time, on='Contact Session ID', how='left')
callbacks = callbacks[callbacks['Activity Start Timestamp'] >= callbacks['first_callback_time']].copy()
callbacks.drop(columns=['first_callback_time'], inplace=True)

# --- Step 6: Create call_date and unique callback instance (per session per day) ---
callbacks['call_date'] = callbacks['Activity Start Timestamp'].dt.date
callbacks['callback_instance'] = (
    callbacks['Contact Session ID'].astype(str) + "_" + callbacks['call_date'].astype(str)
)

# --- Step 7: Flag callback instances that were picked up ---
# A callback is "picked up" if it includes LegalServerScreenPop and has an agent name
picked_up_flags = (
    callbacks.groupby('callback_instance')
    .apply(lambda g: (g['Activity Name'].eq('LegalServerScreenPop').any()) and (g['Agent Name'].notna().any()))
    .reset_index()
    .rename(columns={0: 'picked_up'})
)

# --- Step 8: Flag callback instances that had a retry ---
retry_flags = (
    callbacks.groupby('callback_instance')['Activity Name']
    .apply(lambda x: (x == 'CallbackRetry').any())
    .reset_index()
    .rename(columns={'Activity Name': 'has_callbackretry'})
)

# --- Step 9: Combine both flags into one summary table ---
callback_summary = (
    callbacks[['callback_instance', 'Contact Session ID', 'call_date']]
    .drop_duplicates()
    .merge(picked_up_flags, on='callback_instance', how='left')
    .merge(retry_flags, on='callback_instance', how='left')
)

# --- Step 10: Compute overall summary metrics ---
total_callbacks = len(callback_summary)
picked_up_count = callback_summary['picked_up'].sum()
retry_count = callback_summary['has_callbackretry'].sum()
pickup_rate = picked_up_count / total_callbacks * 100 if total_callbacks > 0 else 0
retry_rate = retry_count / total_callbacks * 100 if total_callbacks > 0 else 0

print(f"Total callback instances: {total_callbacks}")
print(f"Picked-up callbacks: {picked_up_count}")
print(f"Pickup rate: {pickup_rate:.2f}%")
print(f"Callbacks with retries: {retry_count}")
print(f"Retry rate: {retry_rate:.2f}%")

# --- Step 11: Merge pickup and retry flags back into detailed dataset (for Power BI) ---
callbacks = callbacks.merge(picked_up_flags, on='callback_instance', how='left')
callbacks = callbacks.merge(retry_flags, on='callback_instance', how='left')

# --- Step 12: Create a clean summary dataset (one row per callback instance) ---
export_df = (
    callbacks.groupby('callback_instance')
    .agg({
        'Contact Session ID': 'first',
        'call_date': 'first',
        'Activity Start Timestamp': 'min',
        'EP Name': 'first',
        'picked_up': 'max',
        'has_callbackretry': 'max'
    })
    .reset_index()
)

# --- Step 13: Save both detailed and summary versions for Power BI ---
callbacks.to_csv("callbacks_detailed_for_powerbi.csv", index=False)
export_df.to_csv("callback_instances_summary_for_powerbi.csv", index=False)


Total callback instances: 12348
Picked-up callbacks: 9097
Pickup rate: 73.67%
Callbacks with retries: 2645
Retry rate: 21.42%
